# M6 · Lab 2 — Evidently AI Drift Detection
### Module 6 — Monitoring, Testing & Drift Detection | Spine Project: Truck Delay Classification

| | |
|---|---|
| **Duration** | 75 min |
| **Difficulty** | Intermediate |
| **Tools** | Python 3.12.10, `evidently>=0.4,<0.5`, pandas |
| **AWS** | None (local notebook — Lab 4 wires Evidently's output to SNS) |
| **Prerequisite** | The **real** M3 reference frame + model artifacts (provided in `data/` — see Prerequisites) |
| **Builds toward** | Lab 4 (combined pipeline consumes the `dataset_drift` boolean) |
| **Cost** | ₹0 — all local |

---
This is a **notebook lab** — the exploration phase. You'll run drift detection interactively, render Evidently's
HTML report *inline*, tune thresholds, and re-run until the signal is clean. The **production** version of this logic
(a `.py` script that cron / Airflow / SageMaker run on a schedule) is **Lab 5**. Notebook to explore → script to ship.

## 💻 Where to run this — SageMaker (recommended in class), Colab, or local

**In class: the SageMaker Notebook Instance your instructor provisioned** (`m6-truck-delay-monitoring`). Everything is
pre-installed (`evidently`, `great-expectations`, `xgboost`) and the real `data/` folder is already there — just open this
notebook and run.
- **Kernel:** `conda_python3`
- **Data:** `data/reference/final_features.csv` + `data/artifacts/` ship with the labs on the instance (the `M6_DATA_DIR` default is `data`).
- **Auto-stop:** the instance stops itself after ~45 min idle to save cost — your work, installed packages, and saved
  artifacts persist on the EBS volume. Just hit **Start** the next day; **M7 and M8 reuse this same instance.**
- **AWS access (Labs 4/5):** the instance's IAM role already grants `sns:Publish`, so there are **no access keys to configure.**

**Portable fallback (self-paced / post-course):** this notebook also runs on **Google Colab** or **local Jupyter** — the
setup cell below detects the environment and pip-installs what's missing. On Colab, upload the `data/` folder (or set
`M6_DATA_DIR`). Locally, use **Python 3.12.10**. The data + model are the *real* M3 artifacts either way — nothing synthetic.

## Learning Objectives
By the end of this lab you will be able to:
1. Define **data drift**, **concept drift**, and **label drift** with concrete Truck Delay examples.
2. Choose the right Evidently **statistical test** per feature type (Wasserstein for numeric, Jensen-Shannon for categorical).
3. Build an Evidently `Report` with `DataDriftPreset` + `TargetDriftPreset` and read both the HTML and the JSON.
4. Generate **predictions with the real M3 XGBoost model** and detect **prediction drift**.
5. Tune drift thresholds and reason about **alert fatigue** / false-positive rate.
6. Extract a single `dataset_drift: bool` so Lab 4's alerter has something to branch on.

## Business Context
Three months after Truck Delay went live, Priya's team watched the weekly F1-score slide from **0.67 → 0.61**.
Nothing in the code changed. Nothing in the model changed. What changed: the **monsoon arrived two weeks early**, and
the *distribution* of `route_avg_precip` and `route_description` shifted toward "heavy rain" categories that were rare
in the training set. The model wasn't broken — its training data was no longer representative of reality.

This is **data drift**, the most common reason production ML silently degrades. You can't fix what you can't see.
**Evidently** sees it — by computing the statistical distance between the *training* distribution (the **reference**)
and the *live* distribution (the **current**) on every feature, every batch.

## Prerequisites — we reuse the **real** M3 artifacts (nothing synthetic here)

This lab does **not** rebuild a makeshift dataset. The reference distribution and the model are the genuine outputs
of Modules 3–4, provided alongside this notebook in `Module 6/labs/data/`:

| File | What it is | Where it came from |
|---|---|---|
| `data/reference/final_features.csv` | **12,308 × 37** training feature frame — the *reference* distribution | M3 **Lab B** (EDA + Feature Engineering). Regenerated from the committed raw CSVs via `Module 3/labs/regenerate_final_features.py` and **round-trip-validated** against the model (acc 0.93). |
| `data/artifacts/xgboost_model.pkl` | The trained Truck Delay classifier | M3 **Lab C** (Model Training + MLflow) |
| `data/artifacts/{encoder,scaler}.pkl` | The fitted one-hot encoder + scaler | M3 **Lab C** |
| `data/artifacts/model_metadata.json` | Feature names + column groupings (128-feature schema) | M3 **Lab C** |

> **If you jumped straight to M6:** you still use these real files — they ship with the lab. We do **not** fabricate a
> stand-in model or features. The *only* synthetic thing in this lab is the **drifted "current" batch** in Step 3, because
> there is no live production traffic in a classroom — and simulating drift *is the whole point* of a drift-detection lab.

In [ ]:
# ── Environment setup (Google Colab OR local Jupyter) ─────────────────────────
# Evidently 0.5 changed the report-builder API in incompatible ways; we pin the 0.4 line.
import sys, subprocess, os

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import evidently  # noqa
    print("evidently already installed:", evidently.__version__)
except ImportError:
    print("Installing evidently + deps …")
    _pip("evidently>=0.4,<0.5", "pandas==2.2.0", "numpy<2.0", "scikit-learn==1.4.0",
         "xgboost==2.0.3", "joblib==1.3.2")

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

In [ ]:
# ── Imports + paths ───────────────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd

# DATA_DIR points at the folder that ships with this lab.
# Local Jupyter: the notebook's own folder. Colab: upload the `data/` folder, or mount Drive.
DATA_DIR = os.environ.get("M6_DATA_DIR", "data")
REF_CSV   = os.path.join(DATA_DIR, "reference", "final_features.csv")
META_JSON = os.path.join(DATA_DIR, "reference", "feature_metadata.json")
ART_DIR   = os.path.join(DATA_DIR, "artifacts")

assert os.path.exists(REF_CSV), (
    f"Reference frame not found at {REF_CSV}. Copy Module 6/labs/data/ next to this notebook, "
    "or set M6_DATA_DIR. (Do NOT substitute synthetic data — use the real M3 reference frame.)"
)
os.makedirs("artifacts_out", exist_ok=True)   # where this lab writes its outputs
print("Reference CSV:", REF_CSV)

## Step 1 · Concepts — three kinds of drift
Get the vocabulary right — it's the #1 monitoring interview question.

| Kind | One-line definition | Truck Delay example | Caught by |
|---|---|---|---|
| **Data drift** (covariate shift) | input distribution `P(X)` changes; `P(y∣X)` unchanged | more heavy-rain routes than in training | Evidently `DataDriftPreset` |
| **Concept drift** | `P(y∣X)` changes — same inputs, different outcome | new highway tolls change timing, so "short + low-traffic" no longer means on-time | needs fresh labels — hardest to catch |
| **Label/prior drift** | `P(y)` changes — overall delay rate shifts | delay rate climbs 35% → 60% in monsoon | Evidently `TargetDriftPreset` |

> **Practical pattern:** catch **data + label drift in real time** as an early-warning signal (no labels needed), then
> run a periodic ground-truth-aware evaluation (M8 capstone) to *confirm* concept drift. By the time concept drift is
> measurable you've already been serving bad predictions for weeks — so the cheap distribution signal is what pages you.

## Step 2 · Load the real reference frame

In [ ]:
ref = pd.read_csv(REF_CSV)
with open(META_JSON) as f:
    fmeta = json.load(f)

CONTINUOUS  = fmeta["continuous_features"]            # 27
CATEGORICAL = fmeta["categorical_features"]           # 9 (incl. accident, ratings, is_midnight)
TARGET      = fmeta["target"]                         # "delay"

print(f"Reference shape : {ref.shape}      (expected 12308 x 37)")
print(f"Target rate     : {ref[TARGET].mean():.4f}  (delay=1)")
print(f"Continuous ({len(CONTINUOUS)}) | Categorical ({len(CATEGORICAL)}) | Target: {TARGET}")
ref.head(3)

In [ ]:
# Quick profile so you know what 'normal' looks like before we drift it
print("Numeric summary (first 8):")
display(ref[CONTINUOUS].describe().T[["mean", "std", "min", "max"]].head(8).round(2))
print("\nCategorical cardinalities:")
for c in CATEGORICAL:
    print(f"  {c:20s} nunique={ref[c].nunique():3d}  top={ref[c].value_counts().index[0]}")

## Step 3 · Build a realistic "current" (production) batch
There's no live production traffic in class, so we **synthesise one batch** that mimics the monsoon drift Priya's team
actually saw. This is the **only** synthetic step in the lab, and it's deliberate: a drift detector with nothing drifting
is untestable.

We sample real rows from the reference and **perturb a realistic subset of features** — leave the rest untouched so the
report has clean signal to find.

In [ ]:
def make_drifted_batch(reference: pd.DataFrame, n: int = 2000, seed: int = 42) -> pd.DataFrame:
    '''Sample n real rows and inject realistic monsoon-season drift. Functional, no side effects.'''
    rng = np.random.default_rng(seed)
    cur = reference.sample(n=n, random_state=seed).reset_index(drop=True).copy()

    # 1) Monsoon: precipitation + humidity climb on route AND both cities
    for p in ("route", "origin", "dest"):
        cur[f"{p}_avg_precip"]   = cur[f"{p}_avg_precip"]   * rng.uniform(1.3, 1.8, n)
        cur[f"{p}_avg_humidity"] = (cur[f"{p}_avg_humidity"] + rng.normal(8, 2, n)).clip(0, 100)

    # 2) Fleet aging: trucks ~0-3 years older than the training fleet
    cur["truck_age"] = (cur["truck_age"] + rng.integers(0, 4, n)).clip(1, 30)

    # 3) Weather descriptions: more heavy/moderate rain than training saw
    heavy = rng.random(n) < 0.30
    cur.loc[heavy, "route_description"] = rng.choice(
        ["Heavy rain", "Moderate rain", "Patchy rain possible"], size=heavy.sum())

    # 4) Label/prior drift: monsoon => more delays (flip ~15% of on-time rows)
    flip = (cur[TARGET] == 0) & (rng.random(n) < 0.18)
    cur.loc[flip, TARGET] = 1
    return cur

current = make_drifted_batch(ref, n=2000, seed=42)
print(f"Current batch: {current.shape}")
print(f"  delay rate  reference={ref[TARGET].mean():.3f}  current={current[TARGET].mean():.3f}")
print(f"  precip mean reference={ref['route_avg_precip'].mean():.2f}  current={current['route_avg_precip'].mean():.2f}")

## Step 4 · Tell Evidently what each column means (`ColumnMapping`)
Evidently needs to know which columns are numeric vs categorical and which is the target. We read this straight from the
**real** `feature_metadata.json` — no hand-typed column lists that could drift out of sync with the model.

In [ ]:
from evidently import ColumnMapping

column_mapping = ColumnMapping(
    target=TARGET,
    prediction=None,                 # we add predictions in Step 7
    numerical_features=CONTINUOUS,   # 27 continuous
    categorical_features=CATEGORICAL # 9 categorical (incl. accident/ratings/is_midnight)
)
print("Numerical :", len(column_mapping.numerical_features))
print("Categorical:", len(column_mapping.categorical_features))
print("Target    :", column_mapping.target)

## Step 5 · Run the Evidently report (data drift + target drift)

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset

report = Report(metrics=[DataDriftPreset(), TargetDriftPreset()])
report.run(reference_data=ref, current_data=current, column_mapping=column_mapping)
print("Report computed.")

In [ ]:
# Render the interactive HTML report INLINE in the notebook (the whole point of exploring in a notebook)
report.show(mode="inline")

**What you're looking at**
- A top summary card with the **dataset-level drift flag** and the share of drifted columns.
- One card per column: reference vs current distributions side by side, the **statistical test** used
  (Wasserstein for numeric, Jensen-Shannon for categorical), the score and the threshold.
- A **target-drift** card for `delay` showing the prior shift you injected.

## Step 6 · Extract the machine-readable signal (what Lab 4 will consume)

In [ ]:
rd = report.as_dict()
data_drift = rd["metrics"][0]["result"]          # DatasetDriftMetric / DataDriftTable

dataset_drift = bool(data_drift["dataset_drift"])
drift_share   = float(data_drift["drift_share"])
n_drifted     = data_drift["number_of_drifted_columns"]
n_total       = data_drift["number_of_columns"]

drifted_cols = [c for c, r in data_drift["drift_by_columns"].items() if r["drift_detected"]]

print(f"dataset_drift : {dataset_drift}")
print(f"drift_share   : {drift_share:.2%}   ({n_drifted}/{n_total} columns)")
print(f"drifted cols  : {drifted_cols}")

In [ ]:
# Per-column detail — the alert payload Lab 4 publishes to SNS
print(f"{'column':24s} {'test':14s} {'score':>8s} {'thresh':>8s}  drift")
for col, r in data_drift["drift_by_columns"].items():
    if r["drift_detected"]:
        print(f"{col:24s} {r['stattest_name']:14s} {r['drift_score']:8.4f} "
              f"{r['stattest_threshold']:8.4f}  {'✓' if r['drift_detected'] else ''}")

## Step 7 · Prediction drift with the **real** XGBoost model
Data drift answers "did the inputs move?". **Prediction drift** answers "did the model's *outputs* move?" — often the
first thing an on-call engineer notices. We load the genuine M3 model + encoder + scaler and score both frames using the
**exact** inference preprocessing from M3 Lab E / M4 (scale → one-hot → passthrough → reorder to 128 features).

In [ ]:
import joblib

_model   = joblib.load(os.path.join(ART_DIR, "xgboost_model.pkl"))
_encoder = joblib.load(os.path.join(ART_DIR, "encoder.pkl"))
_scaler  = joblib.load(os.path.join(ART_DIR, "scaler.pkl"))
with open(os.path.join(ART_DIR, "model_metadata.json")) as f:
    mmeta = json.load(f)

M_CONT = mmeta["continuous_cols"]            # 27
M_CAT  = mmeta["categorical_cols"]           # 6 (one-hot)
M_BIN  = mmeta["binary_ordinal_cols"]        # 3 (passthrough)
M_FEATS = mmeta["feature_names"]             # 128, in model order

def predict_delay(df: pd.DataFrame) -> np.ndarray:
    '''Replicate M3 inference: scale continuous -> one-hot categoricals -> passthrough binary -> reorder -> predict.'''
    x_cont = pd.DataFrame(_scaler.transform(df[M_CONT]), columns=M_CONT)
    x_cat  = pd.DataFrame(_encoder.transform(df[M_CAT]),
                          columns=_encoder.get_feature_names_out(M_CAT))
    x_bin  = df[M_BIN].reset_index(drop=True)
    x = pd.concat([x_cont, x_cat, x_bin], axis=1)[M_FEATS]
    return _model.predict(x)

ref_pred = predict_delay(ref)
cur_pred = predict_delay(current)
print(f"Predicted delay rate  reference={ref_pred.mean():.3f}  current={cur_pred.mean():.3f}")

In [ ]:
# Feed predictions back into Evidently to measure PREDICTION drift explicitly
ref_p = ref.copy();     ref_p["prediction"] = ref_pred
cur_p = current.copy(); cur_p["prediction"] = cur_pred

cm_pred = ColumnMapping(
    target=TARGET, prediction="prediction",
    numerical_features=CONTINUOUS, categorical_features=CATEGORICAL,
)
pred_report = Report(metrics=[TargetDriftPreset()])
pred_report.run(reference_data=ref_p, current_data=cur_p, column_mapping=cm_pred)
pred_report.show(mode="inline")

## Step 8 · Tune thresholds — avoid alert fatigue
Evidently's defaults are sensible, but on noisy features they fire too often. Three knobs:

**1 — per-column stattest:** loosen tests on noisy columns.
```python
from evidently.metrics import ColumnDriftMetric
Report(metrics=[DataDriftPreset(stattest="wasserstein", stattest_threshold=0.1),
                ColumnDriftMetric(column_name="ratings", stattest="chisquare", stattest_threshold=0.1)])
```
**2 — what to monitor at all:** only the ~36 real features (above), never the 128 sparse one-hot columns — statistical
drift on a 0/1 indicator is meaningless.

**3 — severity routing** (this is what Lab 4 + Lab 5 do):

In [ ]:
def route_severity(dataset_drift: bool, drift_share: float) -> str:
    '''Map a drift result to an alert severity. Lab 4 + Lab 5 reuse this exact policy.'''
    if drift_share > 0.50:
        return "critical"
    if drift_share > 0.30:
        return "warning"
    if dataset_drift:
        return "info"
    return "none"        # below threshold -> don't page

sev = route_severity(dataset_drift, drift_share)
print(f"drift_share={drift_share:.2%}  ->  severity={sev}")

## Step 9 · Save outputs for the downstream labs (artifact chain)

In [ ]:
# Lab 4 (combined pipeline) and Lab 5 (production script) consume these.
report.save_html("artifacts_out/drift_report.html")
with open("artifacts_out/drift_metrics.json", "w") as f:
    json.dump(rd, f, indent=2, default=str)
current.to_csv("artifacts_out/current_features.parquet".replace(".parquet", ".csv"), index=False)

print("Wrote:")
print("  artifacts_out/drift_report.html      (human-readable)")
print("  artifacts_out/drift_metrics.json     (machine-readable -> Lab 4)")
print("  artifacts_out/current_features.csv    (the simulated batch -> Lab 3 & 4)")

## Verification Checklist
- [ ] `data/reference/final_features.csv` loaded — **12,308 × 37**, target rate ≈ 0.349
- [ ] A drifted `current` batch of 2,000 rows was synthesised (precip/humidity/truck_age/description/delay shifted)
- [ ] The inline Evidently report shows the **dataset-level drift banner** + per-column distributions
- [ ] You extracted `dataset_drift: bool` and `drift_share: float`
- [ ] The **real** XGBoost model scored both frames and you saw prediction drift
- [ ] You can name three drifted columns and explain *why* each drifted
- [ ] `artifacts_out/` holds `drift_report.html`, `drift_metrics.json`, `current_features.csv`

## What's next — Lab 3
Lab 2 catches **statistical** drift (slow distribution shifts). **Lab 3 (Great Expectations)** catches **schema** breaks —
corrupted values, NULL spikes, type mismatches, out-of-range numbers. Drift can be subtle; schema breaks are binary.
Together they cover *both* "the data stopped matching reality" and "the data stopped matching the contract". **Lab 4**
fuses both into the SNS-alerting pipeline; **Lab 5** ships it as a production script.

## Troubleshooting
| Symptom | Fix |
|---|---|
| `ModuleNotFoundError: evidently` | Re-run the setup cell; ensure `evidently>=0.4,<0.5` |
| `AttributeError: ... 'iteritems'` | pandas 2 + old Evidently — pin `evidently>=0.4` |
| Inline report is blank | Re-run `report.run(...)` then `report.show(mode="inline")`; in some Colab setups use `report.save_html(...)` and open the file |
| 0 drift even though you drifted features | The drifted columns must be listed in the `ColumnMapping`; confirm names match `feature_metadata.json` |
| `xgboost` import slow/hangs | First import compiles; run the model cells once and reuse. Prediction drift (Step 7) is optional — core drift works without it |
| Encoder warning "unknown categories ... all zeros" | Expected — `fuel_type/gender/driving_style` have `Unknown` from NaN-fill; `handle_unknown='ignore'` zeros them. Harmless. |